# Descenso del Gradiente desde cero

El **descenso del gradiente** es un algoritmo de optimización iterativo: arranca en un punto y se mueve en la **dirección negativa del gradiente** de la función de costo para llegar a su mínimo. En ML lo usamos para minimizar la función de pérdida cuando no hay solución analítica (regresión logística, redes neuronales, etc.).

Aca lo implementamos **a mano** para una regresión lineal simple (`medv ~ lstat` del dataset Boston) y lo comparamos contra `LinearRegression` de sklearn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

## Datos

`boston_data`: predecimos `medv` (valor mediano de la vivienda). Miramos qué variable correlaciona más.

In [ ]:
data = pd.read_csv('../datasets/boston_data.csv')
data.head()

In [ ]:
sns.heatmap(data.corr())

In [ ]:
abs(data.corr()['medv']).sort_values()

`lstat` es la más correlacionada con `medv` (0.74). La usamos como único predictor.

In [ ]:
sns.scatterplot(x=data.lstat, y= data.medv)

In [ ]:
y=data['medv']
X=pd.DataFrame(data['lstat'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=12)

**Estandarizamos**: con las variables normalizadas el descenso del gradiente converge más rápido (la superficie de costo queda más 'redonda').

In [ ]:
scaler = StandardScaler()
X_train_scl = scaler.fit_transform(X_train)
X_test_scl = scaler.transform(X_test)

## Implementación del descenso del gradiente

Modelo: `ŷ = β1·x + β0`. Costo: MSE. En cada paso actualizamos los betas restando el gradiente por el **learning rate** (`α`):

$$\beta_j \leftarrow \beta_j - \alpha \cdot \frac{\partial \text{MSE}}{\partial \beta_j}$$

In [ ]:
learning_rate = 0.01
x=1000

In [ ]:
class MyGradientDescent():
    def __init__(self, learning_rate):
        self.learning_rate = learning_rate
        self.beta1 = 0
        self.beta0 = 0

    def fit(self, X, y, epochs=100):
        N = len(X)
        history = []

        for e in range(epochs):
            for i in range(N):
                Xi = X[i, :]
                yi = y.iloc[i]

                hi = self.beta1 * Xi + self.beta0
                f = hi - yi

                self.beta1 -= self.learning_rate * 2 / N * f * Xi
                self.beta0 -= self.learning_rate * 2 / N * f

            loss = mean_squared_error(y, (self.beta1 * X + self.beta0))

            if e % 100 == 0:
                print(f"Epoch: {e}, Loss: {loss}")

            history.append(loss)

        return history

    def predict(self, X):
        return self.beta1 * X + self.beta0

In [ ]:
model = MyGradientDescent(learning_rate)

history = model.fit(X_train_scl,y_train,x)

predictions=model.predict(X_test_scl)

La **curva de pérdida** (loss vs epoch) muestra cómo el costo baja rápido y luego se estabiliza al converger al mínimo.

In [ ]:
sns.lineplot(x=range(len(history)), y=history)

In [ ]:
sns.scatterplot(x=X_test_scl[:,0], y=y_test)
sns.lineplot(x=X_test_scl.ravel(), y=predictions.ravel(), color='red')

In [ ]:
mean_squared_error(y_test, predictions)

In [ ]:
print(model.beta1, model.beta0)

## Comparación con sklearn

`LinearRegression` resuelve la regresión de forma **analítica** (mínimos cuadrados). Si el descenso del gradiente converge bien, debería llegar casi a los mismos coeficientes y MSE.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_scl, y_train)
predictions_sklearn = model.predict(X_test_scl)
mean_squared_error(y_test, predictions_sklearn)

In [ ]:
print(model.coef_)
print(model.intercept_)

In [ ]:
sns.scatterplot(x=X_test_scl[:,0], y=y_test)
sns.lineplot(x=X_test_scl.ravel(), y=predictions_sklearn, color='red')

**Coinciden**: el GD a mano llega a `β1 ≈ -6.641`, `β0 ≈ 22.284` (MSE 32.009) y sklearn a `β1 ≈ -6.641`, `β0 ≈ 22.284` (MSE 32.009). El descenso del gradiente encontró el mismo mínimo que la solución analítica, porque la función de costo de la regresión lineal es **convexa**.